# mlplatform - Local Workflow

This notebook walks through the full local workflow:
1. **Setup** - Configure paths and PYTHONPATH
2. **Training** - Run a training pipeline via the Python API
3. **In-process Prediction** - Load the trained model and predict
4. **PySpark Batch Prediction** - Distributed prediction via `mapInPandas`
5. **Artifact Inspection** - View what got saved

**Prerequisites:** `pyyaml`, `scikit-learn`, `pandas`, `joblib`, `pyspark`, `pyarrow` must be installed.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("REPO_ROOT", Path("..").resolve())).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "mlplatform") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "mlplatform"))

VERSION = "notebook_v1"
BASE_PATH = str(REPO_ROOT / "artifacts_notebook")
TRAIN_DAG = str(REPO_ROOT / "template_training_dag.yaml")
PREDICT_DAG = str(REPO_ROOT / "template_prediction_dag.yaml")

print(f"REPO_ROOT:   {REPO_ROOT}")
print(f"VERSION:     {VERSION}")
print(f"BASE_PATH:   {BASE_PATH}")
print(f"TRAIN_DAG:   {TRAIN_DAG}")
print(f"PREDICT_DAG: {PREDICT_DAG}")

## 2. Training

Load the training DAG config, generate synthetic data, build an `ExecutionContext`, and run the trainer.

The trainer (`example_model.train.MyTrainer`) will:
- Split data into train/validation
- Fit a StandardScaler + LogisticRegression
- Save `model.pkl` and `scaler.pkl` via `ctx.save_artifact()`
- Log metrics and register the model version

In [ ]:
import pandas as pd
from sklearn.datasets import make_classification
from mlplatform.config.loader import load_workflow_config
from mlplatform.runner import _build_context, _run_training

workflow = load_workflow_config(TRAIN_DAG)
model_cfg = workflow.models[0]

print(f"Workflow:      {workflow.workflow_name}")
print(f"Pipeline type: {workflow.pipeline_type}")
print(f"Feature name:  {workflow.feature_name}")
print(f"Model name:    {model_cfg.model_name}")
print(f"Module:        {model_cfg.module}")

In [ ]:
ctx = _build_context(workflow, model_cfg, "local", VERSION, BASE_PATH)

X, y = make_classification(n_samples=100, n_features=5, random_state=42)
ctx.optional_configs["train_data"] = {
    "X": pd.DataFrame(X, columns=["f0", "f1", "f2", "f3", "f4"]),
    "y": pd.Series(y),
}

_run_training(model_cfg, ctx)
print("Training complete.")

### Check what artifacts were saved

In [ ]:
import json

base = Path(BASE_PATH)
print("Saved artifacts:")
for p in sorted(base.rglob("*.pkl")):
    print(f"  {p.relative_to(base)}")

registry_path = base / "model_registry.json"
if registry_path.exists():
    with open(registry_path) as f:
        registry = json.load(f)
    print("\nModel registry:")
    print(json.dumps(registry, indent=2))

## 3. In-process Prediction

Load the trained model using the predictor class and run prediction on new data.
The predictor uses `ctx.load_artifact()` -- no manual path construction needed.

In [ ]:
pred_workflow = load_workflow_config(PREDICT_DAG)
pred_model_cfg = pred_workflow.models[0]
pred_ctx = _build_context(pred_workflow, pred_model_cfg, "local", VERSION, BASE_PATH)

from example_model.predict import MyPredictor

predictor = MyPredictor()
predictor.context = pred_ctx
predictor.load_model()
print("Model loaded successfully.")

In [ ]:
X_test, _ = make_classification(n_samples=10, n_features=5, random_state=99)
test_df = pd.DataFrame(X_test, columns=["f0", "f1", "f2", "f3", "f4"])

result = predictor.predict_chunk(test_df)
result

### Prediction with the sample CSV from example_model/data/

In [ ]:
inference_csv = REPO_ROOT / "example_model" / "data" / "sample_inference.csv"
inference_df = pd.read_csv(inference_csv)
print(f"Input ({len(inference_df)} rows):")
display(inference_df)

result_csv = predictor.predict_chunk(inference_df)
print(f"\nPredictions:")
display(result_csv)

## 4. PySpark Batch Prediction

This demonstrates the Spark `mapInPandas` flow used for distributed prediction.
The same code runs locally (here) and on cloud (Dataproc/VertexAI).

Steps:
1. Serialize the prediction config to JSON (as `spark/main.py` expects)
2. Create an input CSV
3. Run `_run_spark_inference()` which starts a local Spark session and applies the predictor per partition

In [ ]:
from mlplatform.spark.config_serializer import write_workflow_config

dist_dir = REPO_ROOT / "dist_notebook"
dist_dir.mkdir(exist_ok=True)

spark_config_path = str(dist_dir / "spark_config.json")
write_workflow_config(pred_workflow, pred_model_cfg, spark_config_path,
                      base_path=BASE_PATH, version=VERSION)

with open(spark_config_path) as f:
    print("Spark config JSON:")
    print(json.dumps(json.load(f), indent=2))

In [ ]:
X_spark, _ = make_classification(n_samples=20, n_features=5, random_state=77)
spark_input = pd.DataFrame(X_spark, columns=["f0", "f1", "f2", "f3", "f4"])
spark_csv_path = str(dist_dir / "spark_input.csv")
spark_input.to_csv(spark_csv_path, index=False)
print(f"Input CSV: {spark_csv_path} ({len(spark_input)} rows)")

In [ ]:
from mlplatform.spark.main import _run_spark_inference

spark_output_path = str(dist_dir / "predictions.parquet")

_run_spark_inference(
    config_path=spark_config_path,
    input_path=spark_csv_path,
    output_path=spark_output_path,
)
print("Spark inference complete.")

In [ ]:
spark_result = pd.read_parquet(spark_output_path)
print(f"Output: {len(spark_result)} rows, columns: {list(spark_result.columns)}")
spark_result.head(10)

## 5. Cleanup

Remove the artifacts and dist directories created during this notebook run.

In [ ]:
import shutil

for d in [Path(BASE_PATH), dist_dir]:
    if d.exists():
        shutil.rmtree(d)
        print(f"Removed {d}")

print("Cleanup complete.")